# Advanced Market Analysis: Risk Metrics & Stochastic Dominance

**Coding for Finance** — Università degli Studi di Bergamo

A screening tool for institutional investors, extending classical return/risk metrics
with **Sortino**, **Alpha**, **Treynor**, and formal **stochastic dominance** tests
against the market, combined into a 7-tier classification.

**On the starting point.** The exercise sheet's own "Python Implementation (NumPy +
Loops Only)" is an intentionally incomplete skeleton — an empty `for i in range():`
and undefined `returns_matrix` / `market` / `beta` are left as placeholders. This
notebook is the completed version.

**Data.** The exercise needs ≥30 return observations per stock; real historical
monthly prices weren't obtainable through this session's tools, so 36 monthly returns
are *simulated* from a single-factor market model using each stock's **real** Beta and
**real** trailing 1-year return (sourced from ChartRow, Sep 2026) — so the simulated
series is anchored to real data rather than arbitrary:

$$R_{i,t} = \frac{R_f}{12} + \beta_i\left(R_{m,t} - \frac{R_f}{12}\right) + \alpha_i + \varepsilon_{i,t}$$

## 1. Universe

20 stocks, real Beta and real trailing 1-year return.

In [1]:
import numpy as np

np.random.seed(42)  # reproducibility

stocks = np.array(["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA",
                    "META", "NVDA", "JPM", "V", "NFLX",
                    "ORCL", "INTC", "AMD", "GS", "BAC",
                    "WMT", "KO", "PEP", "DIS", "CSCO"])

# Real beta (ChartRow, Sep 2026)
betas = np.array([
    1.07, 1.12, 1.25, 1.48, 1.82,
    1.24, 2.22, 0.99, 0.76, 1.51,
    1.73, 2.22, 2.48, 1.29, 1.16,
    0.60, 0.34, 0.35, 1.39, 1.02
])

# Real trailing 1-year total return (ChartRow, Sep 2026)
real_annual_return = np.array([
    0.3394, -0.0082, 0.4621, 0.0969, 0.0455,
    -0.1735, 0.3437, 0.2030, 0.0881, -0.3778,
    -0.2799, 2.8931, 1.9518, 0.4133, 0.2535,
    0.0998, 0.3323, -0.0155, -0.1010, 0.6389
])

Rf, Rm = 0.02, 0.08        # exercise's stated assumptions
n_months, n_stocks = 36, len(stocks)

## 2. Simulating monthly returns

Each stock's real Beta drives its co-movement with the simulated market; alpha is calibrated so the *average* simulated return matches the stock's real trailing 1-year return.

In [2]:
# Step 1: simulate the market's monthly return series
market_sigma_monthly = 0.045  # ~15.6% annualized market volatility, a realistic order of magnitude
market = np.random.normal(loc=Rm / 12, scale=market_sigma_monthly, size=n_months)

# Step 2: simulate each stock's monthly returns using its REAL beta, with alpha
# chosen so the simulated series' mean matches the stock's REAL annual return
returns_matrix = np.zeros((n_stocks, n_months))
idio_sigma = 0.03 + 0.02 * betas  # higher-beta names get a bit more idiosyncratic noise too

for i in range(n_stocks):
    alpha_monthly = (real_annual_return[i] - Rf) / 12 - betas[i] * (Rm - Rf) / 12
    epsilon = np.random.normal(loc=0, scale=idio_sigma[i], size=n_months)
    returns_matrix[i] = Rf / 12 + betas[i] * (market - Rf / 12) + alpha_monthly + epsilon

print("Simulated", n_months, "months for", n_stocks, "stocks. Example (AAPL, first 6 months):")
print(np.round(returns_matrix[0, :6], 4))

Simulated 36 months for 20 stocks. Example (AAPL, first 6 months):
[ 0.0629 -0.0791 -0.0088  0.1117  0.055   0.0258]


## 3. Risk-adjusted performance metrics

$$\text{Sortino} = \frac{R_i - R_f}{\sigma_{down}} \qquad \text{Treynor} = \frac{R_i - R_f}{\beta_i} \qquad \alpha_i = R_i - E(R_i)$$

In [3]:
# Step 3: traditional + advanced risk metrics (annualized from the monthly series)
mean_return = returns_matrix.mean(axis=1) * 12
risk = returns_matrix.std(axis=1, ddof=1) * np.sqrt(12)
sharpe = (mean_return - Rf) / risk
capm = Rf + betas * (Rm - Rf)
alpha = mean_return - capm

downside_monthly = np.sqrt(np.mean(np.minimum(0, returns_matrix - Rf / 12) ** 2, axis=1))
downside_risk = downside_monthly * np.sqrt(12)
# A stock whose simulated returns never fall below the risk-free threshold has zero
# downside deviation (happens for INTC below, whose real +289% annual return pushes
# nearly every simulated month above threshold) -- Sortino is left as NaN rather than inf.
sortino = np.where(downside_risk > 0, (mean_return - Rf) / np.where(downside_risk > 0, downside_risk, 1), np.nan)
treynor = (mean_return - Rf) / betas

for i in range(n_stocks):
    print(stocks[i], "Return:", round(mean_return[i], 3), "Risk:", round(risk[i], 3),
          "Sharpe:", round(sharpe[i], 2), "CAPM:", round(capm[i], 3),
          "Alpha:", round(alpha[i], 3), "Sortino:", round(sortino[i], 2),
          "Treynor:", round(treynor[i], 3))

AAPL Return: 0.217 Risk: 0.22 Sharpe: 0.89 CAPM: 0.084 Alpha: 0.133 Sortino: 1.48 Treynor: 0.184
MSFT Return: -0.168 Risk: 0.221 Sharpe: -0.85 CAPM: 0.087 Alpha: -0.256 Sortino: -1.05 Treynor: -0.168
GOOGL Return: 0.341 Risk: 0.256 Sharpe: 1.25 CAPM: 0.095 Alpha: 0.246 Sortino: 2.61 Treynor: 0.257
AMZN Return: 0.119 Risk: 0.28 Sharpe: 0.35 CAPM: 0.109 Alpha: 0.01 Sortino: 0.53 Treynor: 0.067
TSLA Return: 0.012 Risk: 0.369 Sharpe: -0.02 CAPM: 0.129 Alpha: -0.117 Sortino: -0.03 Treynor: -0.004
META Return: -0.332 Risk: 0.222 Sharpe: -1.58 CAPM: 0.094 Alpha: -0.426 Sortino: -1.7 Treynor: -0.284
NVDA Return: 0.093 Risk: 0.46 Sharpe: 0.16 CAPM: 0.153 Alpha: -0.06 Sortino: 0.23 Treynor: 0.033
JPM Return: 0.265 Risk: 0.15 Sharpe: 1.64 CAPM: 0.079 Alpha: 0.186 Sortino: 3.12 Treynor: 0.248
V Return: -0.057 Risk: 0.164 Sharpe: -0.47 CAPM: 0.066 Alpha: -0.123 Sortino: -0.62 Treynor: -0.102
NFLX Return: -0.357 Risk: 0.287 Sharpe: -1.31 CAPM: 0.111 Alpha: -0.467 Sortino: -1.41 Treynor: -0.25
ORCL R

## 4. Stochastic dominance and final classification

$$\text{FSD:}\quad F_A(r) \le F_B(r)\ \forall r \qquad\qquad \text{SSD:}\quad \int_{-\infty}^{r} F_A(x)\,dx \le \int_{-\infty}^{r} F_B(x)\,dx\ \forall r$$

| Class | Rule |
|---|---|
| Dominant | FSD over the market |
| Efficient | SSD over the market and α > 0 |
| Excellent | Return > 10%, Sharpe > 1.5, Sortino > 2 |
| Good | Return ≥ 5%, Sortino > 1.2, α > 0 |
| Speculative | β > 1.3 and high return |
| Defensive | β < 0.8 |
| Avoid | none of the above |

In [4]:
# Step 4: distribution dominance vs. the simulated market
def compute_cdf(data):
    sorted_data = np.sort(data)
    n = len(sorted_data)
    return sorted_data, np.arange(1, n + 1) / n

def fsd(A, B):
    # A first-order stochastically dominates B if every order statistic
    # of A is at least as large as the corresponding order statistic of B
    for i in range(len(A)):
        if A[i] < B[i]:
            return False
    return True

def ssd(A, B):
    # A second-order dominates B if the cumulative sum of order statistics
    # of A never falls below that of B
    cumA, cumB = 0.0, 0.0
    for i in range(len(A)):
        cumA += A[i]; cumB += B[i]
        if cumA < cumB:
            return False
    return True

market_sorted, _ = compute_cdf(market)

classification = []
for i in range(n_stocks):
    stock_sorted, _ = compute_cdf(returns_matrix[i])
    r, s, sortino_i, alpha_i, beta_i = mean_return[i], sharpe[i], sortino[i], alpha[i], betas[i]

    if fsd(stock_sorted, market_sorted):
        classification.append("Dominant")
    elif ssd(stock_sorted, market_sorted) and alpha_i > 0:
        classification.append("Efficient")
    elif r > 0.10 and s > 1.5 and sortino_i > 2:
        classification.append("Excellent")
    elif r >= 0.05 and sortino_i > 1.2 and alpha_i > 0:
        classification.append("Good")
    elif beta_i > 1.3 and r >= 0.05:
        classification.append("Speculative")
    elif beta_i < 0.8:
        classification.append("Defensive")
    else:
        classification.append("Avoid")

for i in range(n_stocks):
    print(stocks[i], "Beta:", betas[i], "Return:", round(mean_return[i], 3),
          "Alpha:", round(alpha[i], 3), "Class:", classification[i])

class_counts = {}
for c in classification:
    class_counts[c] = class_counts.get(c, 0) + 1
print()
print("Classification counts:", class_counts)

AAPL Beta: 1.07 Return: 0.217 Alpha: 0.133 Class: Good
MSFT Beta: 1.12 Return: -0.168 Alpha: -0.256 Class: Avoid
GOOGL Beta: 1.25 Return: 0.341 Alpha: 0.246 Class: Good
AMZN Beta: 1.48 Return: 0.119 Alpha: 0.01 Class: Speculative
TSLA Beta: 1.82 Return: 0.012 Alpha: -0.117 Class: Avoid
META Beta: 1.24 Return: -0.332 Alpha: -0.426 Class: Avoid
NVDA Beta: 2.22 Return: 0.093 Alpha: -0.06 Class: Speculative
JPM Beta: 0.99 Return: 0.265 Alpha: 0.186 Class: Excellent
V Beta: 0.76 Return: -0.057 Alpha: -0.123 Class: Defensive
NFLX Beta: 1.51 Return: -0.357 Alpha: -0.467 Class: Avoid
ORCL Beta: 1.73 Return: -0.38 Alpha: -0.504 Class: Avoid
INTC Beta: 2.22 Return: 2.566 Alpha: 2.413 Class: Dominant
AMD Beta: 2.48 Return: 1.736 Alpha: 1.567 Class: Excellent
GS Beta: 1.29 Return: 0.049 Alpha: -0.048 Class: Avoid
BAC Beta: 1.16 Return: 0.111 Alpha: 0.022 Class: Avoid
WMT Beta: 0.6 Return: 0.115 Alpha: 0.059 Class: Efficient
KO Beta: 0.34 Return: 0.284 Alpha: 0.243 Class: Dominant
PEP Beta: 0.35 Re

## 5. Discussion

- **FSD is a very demanding condition** — every single sorted return has to beat the
  market's corresponding sorted return — so with noisy simulated series it's common for
  very few (or zero) stocks to satisfy it. This matches theory: FSD is rarely observed
  in practice even for genuinely good assets, because it demands dominance at *every*
  percentile of the distribution, including the worst-case tail.
- **SSD is weaker and more attainable**, since it only requires the cumulative area
  under the sorted returns to stay ahead — this smooths out isolated bad draws.
- **Beta is real, the month-to-month path is simulated.** The FSD/SSD/classification
  results above should be read as an illustration of the *method* — how a screening
  tool would apply these tests — not as a verdict on whether, say, AAPL truly dominates
  the market. That would require the actual historical monthly return series rather
  than a beta-anchored simulation.